# Reproducing the whole study on Colab or Kaggle

*AI-guided triage of marine natural products against a six-target glioma panel.*

This notebook runs **every stage, 01 to 12**, of the reproducibility module end to end on a
free Colab or Kaggle session, and finishes by re-asserting the paper's headline numbers.
It is written for a referee who has the tarball and nothing else, and who wants to check
the study without asking the authors for anything.

It does not reimplement any stage. Every cell below calls the module's own runner,
`./run_stage.sh NN`, which is the same command a reader would type on a laptop. The
retrieval stages reuse `repro/fetch_sources.py` and `repro/host_shim.py`, which are the
verified public-API replacements for the two connector-bound steps of stage 01.

**The full pipeline does not fit in one session, and is not meant to.** It is roughly
four and a half days of CPU wall clock, or about two and a half days with a GPU on the
two control stages. Section 2 sets up persistent storage and section 9 gives an explicit
session-by-session running order. Every stage skips work a previous session already
finished.

---
## 1. Read this first: "1,000 epochs" and "a 1,000-epoch ceiling" are not the same thing

This notebook was asked to run the graph training **at the full 1,000-epoch ceiling, with
no epoch reduction**. It does. But the phrase has two readings and only one of them is
what the published control did, so it is worth being exact before any GPU time is spent.

**What the published convergence control actually did.** It set the epoch **ceiling** to
1,000 and the early-stopping **patience** to 25, restored the networks to reference
capacity, and then let early stopping decide when each run ended. Early stopping fired in
**all fifteen runs, between epochs 34 and 86**. Not one run came within an order of
magnitude of the ceiling. The ceiling was a bound that was never reached, not a training
length.

**What "train for a literal 1,000 epochs" would do instead.** It would keep training past
the validation minimum. The validation-loss curves of AttentiveFP, GraphGPS and D-MPNN
**rise** after their minima, so the extra epochs do not refine the model, they overfit it.
The result would reproduce no published number, and it would cost roughly fifteen to
twenty times the compute to arrive at a worse model.

**So this notebook runs ceiling 1,000 with early stopping patience 25, and does not remove
early stopping.** Relative to the 30-epoch ceiling of the main benchmark, that *is* "no
epoch reduction" - it is the largest ceiling in the study, 33x the main one. What it is
not is a promise to burn 1,000 epochs per run.

**One more distinction, because it decides which stage you are looking at:**

| | stage 05, the main benchmark | stage 09, the convergence control |
|---|---|---|
| epoch ceiling | **30** | **1,000** |
| early-stopping patience | **6** | **25** |
| capacity | paper scale | reference ("full") scale |
| what it produces | the published internal and external ROC-AUCs | the control showing the result is not an under-training artefact |

Stage 05's 30/6 is **not** raised here. Those two numbers *are* the published main
benchmark: internal macro ROC-AUC 0.922 for the random forest down to 0.867 for GATv2,
and the marine external arm. Changing them would change every headline number in the
paper, which is the one thing a reproduction must not do. Stage 09 exists precisely to
answer "what happens if you train to convergence instead", and its answer is the
0.394-0.538 marine band - still chance.

The two values are read from `config/config.yaml` (`control_convergence.epochs_max` and
`.patience`) and passed to the trainer by `stages/legacy/stage09/run_convergence.sh`.
They are printed back to you in section 3 so you can see what this session will use.

---
## 2. What each stage costs, and whether a GPU helps

Wall clock is the module's own per-stage declaration, measured on its reference machine
(8 CPU cores, 8 GB RAM, no GPU). Treat them as planning numbers: a Colab CPU runtime is
in the same class, a Kaggle CPU runtime is usually a little slower, and the network-bound
stages depend entirely on how the public APIs are feeling.

| stage | what it does | CPU | with a GPU | GPU worth it? |
|---|---|---|---|---|
| 01 target discovery | Open Targets / CIViC / STRING / ChEMBL | ~25 min | ~25 min | **no** - API-bound |
| 02 marine library | CMNPD curation | ~45 min | ~45 min | **no** |
| 03 label matrix | ChEMBL retrieval + scaffold split | ~3 h | ~3 h | **no** - API-bound |
| 04 controls | ZINC20 decoys + FDA pairs | ~70 min | ~70 min | **no** - 40 min of it is a download |
| 05 training | 7 model families x 3 seeds | ~26 h | ~26 h | **no, see below** |
| 06 evaluation | internal + three external arms | ~90 min | ~90 min | **no** |
| 07 attribution | exact TreeSHAP + consensus rank | ~4 h | ~4 h | **no** - no GPU path exists |
| 08 screening | 26,754-compound cascade | ~25 min | ~25 min | **no** |
| 09 convergence control | 15 runs to convergence | ~40 h | **~4 h** | **yes** |
| 10 hyperparameter control | 15,930 fits | ~30 h | **~5 h** | **partly** - XGBoost arm only |
| 11 figures | 12 figures, PDF + 300 dpi PNG | ~8 min | ~8 min | **no** |
| 12 verify | 44 assertions | ~2 s | ~2 s | **no** |

**Stage 05 does not use a GPU, whatever the package README says.** This was checked
against the code rather than assumed: `src/train_gnn.py` (the main-benchmark trainer)
declares no `--device` argument and contains no `.to(device)` call anywhere, and
`src/baselines.py` builds `XGBClassifier(tree_method="hist")` with no `device=`. Both run
on CPU on a GPU machine. The README's "~3 h on one modern GPU" for stage 05 is not
reachable with the shipped code. Do not spend GPU quota on stage 05: attach a GPU only
for stages 09 and 10.

**Stage 10 is only half a GPU job.** Its XGBoost arm honours `DEVICE=cuda`; the random
forest arm is scikit-learn on CPU *by design*, because the interpretation chapter
consumes a scikit-learn forest. The optional cuML screening pass is what produces the
~5 h figure.

**Stage 07 has no GPU path at all.** It is exact path-dependent TreeSHAP on a
scikit-learn random forest. Four hours of CPU is the honest cost.

Summing that table: about **107 h (4.5 days)** on CPU throughout, falling to about
**46 h (1.9 days)** once stages 09 and 10 move to a GPU. Stage 05's 26 h is then the
largest single item, which is why the GPU total is not smaller. Stages 09 and 10 are independent controls - skipping both still
reproduces every main-text result, and costs you Fig. S4 and Tables 8 and 9.

---
## 3. Session setup

Run section 3 in full at the start of **every** session, including after a dropped
runtime. It is cheap, and it is what makes the later cells resume instead of restart.

### 3a. Persistent storage, before anything else

Colab recycles runtimes and Kaggle hard-stops a session at **12 hours**. Anything written
to the session's own disk is gone when that happens. Every output, cache and checkpoint
this notebook produces therefore goes to storage that outlives the session: Google Drive
on Colab, `/kaggle/working` on Kaggle (remember to **Save Version** so Kaggle keeps it).

Do not skip this cell and do not start a long stage before it has succeeded.

In [ ]:
import os, sys, json, time, shutil, subprocess, pathlib

def detect_platform():
    """Identify the runtime.

    Colab is tested FIRST and by the presence of its own module. The old order
    tested `pathlib.Path("/kaggle").exists()` first, which reports kaggle on a
    Colab VM that happens to carry a /kaggle directory - and that misdetection is
    silent, sending PERSIST to /kaggle/working, which is not on Drive, so every
    output died with the session. Kaggle is then identified by the environment
    variable its runtime always sets, rather than by a directory anyone can create.
    """
    if "google.colab" in sys.modules:
        return "colab"
    try:
        import google.colab            # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    return "local"

PLATFORM = detect_platform()

# Where results must survive a dropped runtime. On Colab this has to be a Drive
# path: /content is wiped when the session ends. Edit PROJECT to point at your own
# project folder; everything else follows from it.
PROJECT = "Projek/tirta/BMC-cmnpd_screening/glioma_final"

if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = pathlib.Path("/content/drive/MyDrive") / PROJECT
    # The code checkout goes on the fast local disk, not on Drive: unpacking and
    # importing several hundred files over the Drive FUSE mount is slow, and Drive
    # does not carry the POSIX executable bit (see the `bash` note in section 4a).
    WORKDIR = pathlib.Path("/content")
elif PLATFORM == "kaggle":
    PERSIST = pathlib.Path("/kaggle/working/glioma_repro")
    WORKDIR = pathlib.Path("/kaggle/working")
else:
    PERSIST = pathlib.Path.home() / "glioma_repro"
    WORKDIR = pathlib.Path.cwd()

for sub in ("data", "outputs", "checkpoints", "cache", "logs"):
    (PERSIST / sub).mkdir(parents=True, exist_ok=True)

print("platform        :", PLATFORM)
print("persistent root :", PERSIST)
print("session workdir :", WORKDIR)
print("already present :", sorted(p.name for p in (PERSIST / "outputs").iterdir())[:12] or "(nothing yet)")


### 3b. Unpack the module

Put `marine_antiglioma_reproducibility.tar.gz` in the persistent root
(`MyDrive/glioma_repro/` on Colab, or add it as a Kaggle dataset) before running this.
The code checkout goes on the **session** disk because the stage scripts do many small
reads and Drive's filesystem is slow for that; nothing of value is stored there, because
section 3c redirects all outputs to the persistent root.

In [ ]:
TARBALL = PERSIST / "marine_antiglioma_reproducibility.tar.gz"
if not TARBALL.exists():
    raise SystemExit(
        f"{TARBALL} not found.\n"
        "Colab : upload marine_antiglioma_reproducibility.tar.gz into MyDrive/glioma_repro/\n"
        "Kaggle: attach it as a dataset, then copy it to /kaggle/working/glioma_repro/")

REPO = WORKDIR / "marine_antiglioma_reproducibility"
if not REPO.exists():
    subprocess.run(["tar", "xzf", str(TARBALL), "-C", str(WORKDIR)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

print("module root:", REPO)
print("stages     :", sorted(p.name for p in (REPO / "stages").glob("stage[0-9][0-9]_*.py")))

### 3c. Environment variables - set them in Python, not with `!export`

Every `!line` in a notebook is **its own subshell**. `!export FOO=bar` on a line of its
own is therefore forgotten the instant that line finishes, which is a genuinely easy way
to run a twelve-hour stage with the wrong configuration and not notice. Variables set
through `os.environ` are inherited by every `!` line and every subprocess, so that is
what this notebook uses throughout.

Four variables matter:

| variable | what it does |
|---|---|
| `MARINE_REPRO_DATA` | where the large study inputs are looked up first |
| `MARINE_REPRO_OUTPUT` | where **all** stage outputs go - the single most important one for surviving a dropped runtime |
| `GLIOMA_CKPT` | stage 10's per-configuration resume state |
| `EPOCHS` / `PATIENCE` | stage 09's ceiling and patience, read from `config/config.yaml` |

In [ ]:
os.environ["MARINE_REPRO_DATA"]   = str(PERSIST / "data")
os.environ["MARINE_REPRO_OUTPUT"] = str(PERSIST / "outputs")
os.environ["GLIOMA_CKPT"]         = str(PERSIST / "checkpoints")
os.environ["GLIOMA_CACHE"]        = str(PERSIST / "cache")

from repro.config import CFG, out_dir           # noqa: E402
cc = CFG["control_convergence"]
os.environ["EPOCHS"]   = str(cc["epochs_max"])
os.environ["PATIENCE"] = str(cc["patience"])

print("data   ->", os.environ["MARINE_REPRO_DATA"])
print("outputs->", os.environ["MARINE_REPRO_OUTPUT"], "  (resolved:", out_dir() + ")")
print("ckpt   ->", os.environ["GLIOMA_CKPT"])
print()
print("stage 05 main benchmark   : ceiling %d epochs, patience %d  (published, unchanged)"
      % (CFG["training"]["epochs"], CFG["training"]["patience"]))
print("stage 09 convergence ctrl : ceiling %d epochs, patience %d, capacity %s"
      % (cc["epochs_max"], cc["patience"], cc.get("capacity", "full")))
print("                            early stopping decides; the published runs stopped at epochs 34-86")

### 3d. Packages

The module pins its environment in `env/environment.yml` and `env/requirements.txt`.
Colab and Kaggle differ from those pins in ways that matter differently per package:

* **torch / torch-geometric.** The pinned `torch==2.13.0` is the CPU reference wheel.
  **Do not install it over the platform's torch.** Colab and Kaggle ship a CUDA build
  matched to their driver, and replacing it is the usual way to end up with a GPU the
  runtime can no longer see. This notebook installs `torch-geometric` against whatever
  torch the platform provides and leaves torch alone.
* **scikit-learn, xgboost, rdkit, shap.** These decide the reported numbers - the
  selected model is a scikit-learn random forest on RDKit ECFP4 features, and stage 07 is
  TreeSHAP. They are pinned exactly.
* **numpy / pandas / scipy.** Left to the platform. Pinning numpy on Colab forces a
  runtime restart and frequently breaks the preinstalled torch ABI; the module's results
  do not depend on these at the third decimal.

Stage 12 is the arbiter, not this cell: if a pin difference moved a number, the 44
assertions say so by name.

In [ ]:
PINNED = ["scikit-learn==1.9.1", "xgboost==3.4.1", "rdkit==2026.03.6", "shap==0.52.0",
          "pyarrow==25.0.0", "networkx==3.6.1", "statsmodels==0.15.0", "seaborn==0.13.2",
          "torch-geometric==2.8.0.post1", "PyYAML"]

import importlib.util
def version_of(mod):
    try:
        import importlib.metadata as md
        return md.version(mod)
    except Exception:
        return "absent"

print("before:", {m: version_of(m) for m in
                  ["torch", "scikit-learn", "xgboost", "rdkit", "shap", "numpy", "pandas"]})

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PINNED,
                   capture_output=True, text=True)
print(r.stdout[-1500:] or "pip: quiet install finished")
if r.returncode:
    print("PIP FAILED:\n", r.stderr[-2000:])

print("after :", {m: version_of(m) for m in
                  ["torch", "torch-geometric", "scikit-learn", "xgboost", "rdkit", "shap"]})

### 3e. Is there a GPU, really?

Probe the **driver**, with `nvidia-smi -L`. Do not probe by attempting a GPU train:
xgboost 2.1 and later respond to `device="cuda"` on a CPU-only machine by falling back to
CPU **with a warning rather than an exception**, so a training call that "succeeded" tells
you nothing. That exact mistake already produced one false "GPU present" report in this
project.

In [ ]:
def gpu_present():
    """Ask the driver. Returns (bool, description).

    nvidia-smi is ABSENT, not merely unhelpful, on a CPU-only runtime - subprocess.run
    then raises FileNotFoundError rather than returning non-zero. Catching that is not
    hypothetical: an earlier draft of this cell died exactly there on a CPU machine and
    took every later cell down with it, because HAS_GPU was never defined.
    """
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=30)
    except (FileNotFoundError, OSError, subprocess.TimeoutExpired) as e:
        return False, f"nvidia-smi not available ({type(e).__name__}) - treating as CPU-only"
    if r.returncode == 0 and "GPU" in r.stdout:
        return True, r.stdout.strip().splitlines()[0][:72]
    return False, "no CUDA device visible to the driver"

HAS_GPU, GPU_DESC = gpu_present()
print("GPU:", GPU_DESC)
print()
if HAS_GPU:
    print("Useful for stages 09 and 10 only. Stages 01-08, 11 and 12 are CPU or network")
    print("bound and will not go faster - if this session is only going to run those,")
    print("release the accelerator and save your quota.")
else:
    print("Stages 01-08, 11 and 12 run here unchanged. Stages 09 (~40 h) and 10 (~30 h)")
    print("will not finish in a free CPU session: attach a GPU for those.")
    print("  Colab : Runtime -> Change runtime type -> T4 GPU")
    print("  Kaggle: Settings -> Accelerator -> GPU")

### 3f. The stage runner, with a resume ledger

`rs(n)` is a thin wrapper around the module's own `./run_stage.sh n`: it streams the
output, records the wall clock into the persistent root, and writes a `.done_stageNN`
marker so that re-running the cell in a later session reports the stage as already
finished instead of redoing it. `rs(n, "--check")` and `rs(n, force=True)` do what you
would expect, and the literal shell command is printed first so you can always run it by
hand instead.

In [ ]:
LEDGER = PERSIST / "logs" / "session_ledger.jsonl"

PROBE_ARGS   = ("--check", "--list")                 # resolve/print only, nothing runs
PARTIAL_ARGS = PROBE_ARGS + ("--smoke", "--only")    # runs, but does not complete the stage

def rs(n, *args, force=False, mark=None):
    """Run the module's own ./run_stage.sh n [args...], with a resume marker.

    A stage is marked complete only when the WHOLE stage ran and exited 0. A --only
    group, a --smoke rehearsal or a --check never marks it, so finishing stage 05
    group A does not convince a later session that stage 05 is done. mark=False
    suppresses marking explicitly - the stage-10 single-target rehearsal uses it.
    """
    tag = f"{int(n):02d}"
    marker = pathlib.Path(os.environ["MARINE_REPRO_OUTPUT"]) / f".done_stage{tag}"
    probe   = any(a in PROBE_ARGS for a in args)
    partial = any(a in PARTIAL_ARGS for a in args)
    if mark is None:
        mark = not partial
    if marker.exists() and not partial and not force:
        print(f"stage {tag}: already completed on {marker.read_text().strip()}")
        print(f"            rs({int(n)}, force=True) to run it again")
        return 0
    # `bash run_stage.sh`, never `./run_stage.sh`: a tarball unpacked onto Google
    # Drive loses the POSIX executable bit, and `chmod +x` does not stick on Drive.
    cmd = ["bash", "run_stage.sh", tag, *args]
    print("$", " ".join(cmd), flush=True)
    t0 = time.time()
    p = subprocess.run(cmd, cwd=str(REPO), env=dict(os.environ, PYTHONPATH=str(REPO)))
    dt = time.time() - t0
    MEANING = {0: "ok", 1: "declared inputs missing (a report, not a crash)",
               2: "refused to run: inputs missing or stage not executable here"}
    print(f"\n[stage {tag}] exit {p.returncode} ({MEANING.get(p.returncode, 'error')}) "
          f"in {dt/60:.1f} min", flush=True)
    if p.returncode and not probe:
        print(f"            scroll up for the step that failed; "
              f"rs({int(n)}, '--check') lists what is missing", flush=True)
    if not probe:
        with open(LEDGER, "a") as fh:
            fh.write(json.dumps(dict(stage=tag, args=list(args), rc=p.returncode,
                                     minutes=round(dt / 60, 2), platform=PLATFORM,
                                     gpu=bool(globals().get("HAS_GPU", False)),
                                     marked=bool(mark and p.returncode == 0),
                                     when=time.strftime("%Y-%m-%d %H:%M"))) + "\n")
        if p.returncode == 0 and mark:
            marker.write_text(time.strftime("%Y-%m-%d %H:%M"))
    return p.returncode


def ledger():
    if not LEDGER.exists():
        print("nothing run yet"); return
    for line in LEDGER.read_text().splitlines():
        r = json.loads(line)
        print(f"  stage {r['stage']}  rc={r['rc']}  {r['minutes']:8.1f} min  "
              f"{r['when']}  {r['platform']}{' GPU' if r['gpu'] else ''}  "
              f"{'[complete]' if r.get('marked') else '[partial] '}  {' '.join(r['args'])}")

ledger()

---
## 4. The ten-minute rehearsal - do this before committing hours

Four cheap checks that between them exercise retrieval, training and figure rendering.
They are here so that a platform problem costs you ten minutes rather than surfacing
three hours into stage 03. If all four pass, the long stages will run.

**4a. The module's own smoke test** (~1-2 min). Imports, config parsing, all twelve stages' `--check`, two figure renders, and the full 44-assertion verification against the tables that ship inside the package. A reference run is `44 PASS, 0 FAIL, 0 SKIP`.

In [ ]:
subprocess.run(["bash", "smoke_test.sh"], cwd=str(REPO),
               env=dict(os.environ, PYTHONPATH=str(REPO)))

**4b. Retrieval** (~1 min). Reachability of the six services stages 01, 03 and 04 depend on, then one real Open Targets query through `repro/fetch_sources.py` - the same code path stage 01 uses. A `403` or `404` answer to a `HEAD` request is fine; `UNREACHABLE` is not.

In [ ]:
import urllib.request, urllib.error
for name, url in [("Open Targets", "https://api.platform.opentargets.org/api/v4/graphql"),
                  ("CIViC",        "https://civicdb.org/api/graphql"),
                  ("STRING",       "https://string-db.org/api/json/version"),
                  ("Reactome",     "https://reactome.org/AnalysisService/database/version"),
                  ("ChEMBL",       "https://www.ebi.ac.uk/chembl/api/data/status?format=json"),
                  ("ZINC",         "https://files.docking.org")]:
    try:
        urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=25)
        print(f"  {name:13s} reachable")
    except urllib.error.HTTPError as e:
        print(f"  {name:13s} reachable (HTTP {e.code} to a HEAD request is fine)")
    except Exception as e:
        print(f"  {name:13s} UNREACHABLE - {type(e).__name__}: {str(e)[:60]}")

from repro.fetch_sources import fetch_open_targets
ot = fetch_open_targets({"MONDO_0018177": "glioblastoma"}, n=25, verbose=False)
blk = ot["MONDO_0018177"]
print(f"\n  live query: {len(blk['rows'])} of {blk['count']} glioblastoma associations; "
      f"top symbols {[r['target']['approvedSymbol'] for r in blk['rows'][:6]]}")

**4c. Training** (~30 s CPU, faster on GPU). The stage-09 smoke path builds synthetic batches shaped like the real ones, constructs all five architectures at reference capacity, and **times real training batches**. The seconds-per-epoch it prints is the number to size stage 09 with on *this* machine - multiply by the epochs you expect early stopping to reach (the published runs stopped at 34-86) and by 15 runs. It reads none of the study data, so it works before anything has been downloaded.

In [ ]:
rs(9, "--smoke")

**4d. Figures and verification** (~15 s). Renders the three figures that need only what ships in the package, then re-asserts the headline numbers. Note that stage 12 reports `SKIP` for the figure-resolution check until stage 11 has written figures into *this* output directory - `SKIP` is never `PASS`.

In [ ]:
rs(11, "--only", "1", "7", "S1")
rs(12, mark=False)

**If the rehearsal fails.** 4a failing on an import is a package problem - re-run 3d and
read its output. 4b failing on one service is usually transient; Reactome in particular
sits behind Cloudflare and refuses some automated clients outright, which costs you only
the optional pathway-enrichment table (the six-target panel does not depend on it) - do
not disguise the client to get around it. 4c failing means torch or torch-geometric did
not install against the platform's torch. 4d failing is the serious one: it means the
package cannot reproduce its own shipped numbers, and nothing further is worth running.

---
## 5. What you must download yourself

`data/shipped/` (61 result tables, 9.2 MB) comes inside the tarball and is enough for
stages 11 and 12. Everything else falls into three groups, listed authoritatively with
sizes and SHA-256 in `config/inputs_manifest.tsv`:

**1. The CMNPD vendor release - you must fetch this, and nobody can send it to you.**
`CMNPD_1.0_calc_prop.tsv` and `CMNPD_1.0.sdf`, the Comparative Marine Natural Products
Database release 1.0. The licence does not permit redistribution, CMNPD has no public API,
and there is no automated path: register at **<https://www.cmnpd.org>**, download release
1.0 by hand, and drop both files anywhere under `MARINE_REPRO_DATA`. Only stages 02 and 03
read them. Without them, stage 02 cannot run and stage 03 stops at its last step.

**2. The study data products** (21 files, ~173 MB): the curated marine library, the ChEMBL
label matrix, the decoy set, the applicability-domain table, the model bundle, the SHAP
tensors, the scored library. These are a *convenience, not a dependency* - each is
regenerated by the stage named in its manifest row, so running stages 01-08 produces them.
Download the deposit alongside the paper and unpack into `MARINE_REPRO_DATA` to skip
ahead.

**3. Live API data.** Stages 01, 03 and 04 fetch from Open Targets, CIViC, STRING,
Reactome, QuickGO, ChEMBL and ZINC20 at run time. Those services evolve. Cached responses
from the original run are in group 2 and are preferred when present, which is what makes
the pipeline deterministic; delete them to force a live re-fetch, and read section 10 on
what may legitimately move afterwards.

The next cell reports, stage by stage, exactly which inputs resolve on this machine.

In [ ]:
subprocess.run(["bash", "run_all.sh", "--check"], cwd=str(REPO),
               env=dict(os.environ, PYTHONPATH=str(REPO)))

---
## 6. The stages

Each stage gets three cells: what it is, a `--check` that resolves its inputs and runs nothing, and the run itself. Run the `--check` first - it costs a second and it is the difference between discovering a missing input now and discovering it two hours in. Any stage can be run on its own, in any session, as long as section 3 has been re-run first.

### Stage 01 - target discovery from the protein-interaction network

~25 min, CPU + network. **No GPU.**

Open Targets and CIViC seeds, a STRING subnetwork at combined score >= 0.70, greedy-modularity modules, centrality, pathway enrichment, ChEMBL tractability, and the six-target panel EGFR / PDGFRA / PIK3CA / KDR / CDK6 / IDH1.

Two of this stage's nine steps were originally run through a hosted connector that does not exist outside the authors' environment. `repro/fetch_sources.py` replaces those calls with the services' public APIs and `repro/host_shim.py` feeds the results to the study's own selection code, so the logic that chose the 186 seed genes is executed rather than reimplemented. Section 6a below drives that path; the remaining steps run through the normal runner.

A third gap: step 07 reads `handoff/chembl_targets.json`, which no script in the module writes - it was produced outside the shipped pipeline. `fetch_chembl_targets()` rebuilds it.

In [ ]:
rs(1, "--check")

**6a. Stage 01 steps 01-03 through the public APIs.** The study's own script derives the seed-symbol list and only then asks for the network, so it is run in two passes: the first stops at the STRING call, we fetch the network for exactly the symbols it produced, and the second completes. The shim raises rather than returning an empty network, so a missing payload cannot silently shrink the seed set.

In [ ]:
from repro.fetch_sources import (fetch_open_targets, fetch_civic, fetch_string,
                                 fetch_chembl_targets, GLIOMA_DISEASES, CIVIC_DISEASES)
from repro.host_shim import HostShim, run_legacy

WORK = pathlib.Path(os.environ["MARINE_REPRO_OUTPUT"]) / "stage01_target_discovery"
(WORK / "handoff").mkdir(parents=True, exist_ok=True)
SCRIPT = str(REPO / "stages/legacy/stage01/03_fetch_string_network.py")

ot    = fetch_open_targets(GLIOMA_DISEASES, n=200)
civic = fetch_civic(CIVIC_DISEASES)

try:                                        # pass 1 - selection logic, stops at STRING
    run_legacy(SCRIPT, HostShim(open_targets=ot, civic=civic), workdir=str(WORK))
except RuntimeError as e:
    assert "no STRING payload" in str(e), e
    print("pass 1 complete - selection logic ran, STRING call reached")

symbols = json.load(open(WORK / "handoff" / "seed_symbols.json"))
print(f"seed genes selected by the study's own code: {len(symbols)}  (published: 186)")

net = fetch_string(symbols, species=9606, required_score=700)
run_legacy(SCRIPT, HostShim(open_targets=ot, civic=civic, string=net), workdir=str(WORK))

run_legacy(str(REPO / "stages/legacy/stage01/04_detect_modules.py"),
           HostShim(open_targets=ot, civic=civic, string=net), workdir=str(WORK))

r = subprocess.run([sys.executable, str(REPO / "stages/legacy/stage01/05_build_centrality_table.py")],
                   cwd=str(WORK), env=dict(os.environ, PYTHONPATH=str(REPO)),
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-500:]

# handoff/chembl_targets.json is read by step 07 but written by no script in the module.
import pandas as pd
cen = pd.read_csv(WORK / "handoff" / "centrality.csv")
json.dump(fetch_chembl_targets(cen.gene_symbol.tolist()),
          open(WORK / "handoff" / "chembl_targets.json", "w"))
print(f"centrality table: {len(cen)} genes; chembl_targets.json rebuilt")

**6b. Stage 01 steps 07-09** - the ChEMBL compound counts per hub and the panel selection. This is the part that produces Table 1, and it is run on its own because the optional enrichment step in 6c declares inputs that may legitimately be absent: the runner refuses an entire `--only` group when any step in it has a missing input, so bundling them would let a Reactome outage block the panel.

In [ ]:
rs(1, "--only", "07_fetch_chembl_hub_counts", "08_select_target_panel",
      "09_export_centrality_table")

panel = pd.read_csv(WORK / "glioma_target_panel.csv")
print("\nselected panel:", panel.iloc[:, 0].tolist())
print("published    : ['EGFR', 'PDGFRA', 'PIK3CA', 'KDR', 'CDK6', 'IDH1']")

**6c. Pathway enrichment - optional, and skippable.** Reactome and GO feed the descriptive pathway table. Neither is an input to target selection: `08_select_target_panel.py` reads only `chembl_counts.json` and `centrality.csv`. **Reactome refuses some automated clients** - it sits behind Cloudflare and answers with error 1010 from some networks whatever the request looks like. Do not disguise the client to get around it; either run from a network it accepts, or paste each module's gene list into the Reactome analysis page by hand. GO goes through UniProt and QuickGO and is reliable.

In [ ]:
from repro.fetch_sources import fetch_go, fetch_reactome
DATA = pathlib.Path(os.environ["MARINE_REPRO_DATA"])
mod_of = json.load(open(WORK / "handoff" / "modules.json"))
mod_genes = {}
for gene, m in mod_of.items():
    mod_genes.setdefault(str(m), []).append(gene)
print("modules:", {k: len(v) for k, v in sorted(mod_genes.items())})
print("published: 4 modules of 64/56/43/8 genes, Q = 0.380, plus 15 unassigned")

json.dump(fetch_go(sorted(mod_of)), open(DATA / "go_raw.json", "w"))
try:
    json.dump(fetch_reactome(mod_genes), open(DATA / "reactome_raw.json", "w"))
    rs(1, "--only", "06_pathway_enrichment")
except RuntimeError as e:
    print("Reactome unavailable from this network:", str(e)[:220])
    print("Skipped. The pathway table is the only casualty; the panel above")
    print("was selected without it.")

### Stage 02 - marine library curation

~45 min, CPU. **No GPU.**

Standardisation, element and size QC, InChIKey de-duplication and property annotation of CMNPD 1.0: 31,561 records in, 26,754 curated compounds out.

**Needs the two CMNPD vendor files from section 5.** If they are absent this stage reports exactly that and runs nothing; the curated library ships in the data deposit, so you can carry on without re-deriving it.

In [ ]:
rs(2, "--check")

In [ ]:
rs(2)

### Stage 03 - the ChEMBL label matrix and the scaffold-disjoint split

~3 h, CPU + network. **No GPU** - this is API latency, not computation.

Two-pass ChEMBL activity retrieval, filtering, censored-bound recovery, standardisation, the 32,403 x 6 label matrix, the chemical-space projection, the applicability domain and the Bemis-Murcko split.

The single longest network stage. Run it in a session of its own. Each step writes its own outputs under the persistent output directory, so a session that dies partway through does not lose the steps that already finished - re-run the cell and use `--only` to resume from the step that was interrupted (`rs(3, '--list')` names them).

In [ ]:
rs(3, "--check")

In [ ]:
rs(3)

### Stage 04 - control sets

~70 min, CPU + network (about 40 min of it is the ZINC20 pool download). **No GPU.**

Positive controls are ChEMBL `max_phase 4` drug-target pairs forced into the test partition; negative controls are 45,213 ZINC20 decoys matched on six DUD-E properties with ECFP4 Tanimoto < 0.35 to every active.

In [ ]:
rs(4, "--check")

In [ ]:
rs(4)

### Stage 05 - training all seven model families

~26 h, CPU. **No GPU** - see section 2; the main-benchmark trainer has no device argument and no `.to(device)` call, so a GPU session buys nothing here.

Graph cache and flat feature build, then five graph architectures x three seeds plus the two fingerprint ensembles, under the published **30-epoch ceiling with patience 6**. Those two numbers are the main benchmark and are not raised - section 1 explains why the 1,000 ceiling belongs to stage 09 instead.

**This stage does not fit in one Kaggle session and must be split.** It is seven steps; run them in groups with `--only`. The pretraining step resumes: each of the fifteen runs writes `metrics/<tag>_run.json` when it completes, and the scheduler skips any run whose marker already exists, so re-running the cell in a new session picks up where the last one stopped.

In [ ]:
rs(5, "--check")
rs(5, "--list")

Stage 05 split into three session-sized groups. Run group A, then B (possibly over two sessions - it resumes), then C. `rs(5)` with no arguments runs all seven steps in order if you have ~26 h to give it.

In [ ]:
# group A - features and caches, ~4.5 h
rs(5, "--only", "build_graph_cache", "flatten_arrays", "fingerprint_features")

In [ ]:
# group B - the fifteen pretraining runs, ~20 h. RESUMABLE: re-run this cell in
# a new session and completed runs are skipped by their metrics/<tag>_run.json.
rs(5, "--only", "pretrain_graph_networks")

In [ ]:
# group C - fingerprint baselines, ablation, final forest, ~5.5 h
rs(5, "--only", "fit_baselines", "ablation_finetune", "persist_final_rf")

### Stage 06 - evaluation, external validation and model selection

~90 min, CPU. **No GPU.**

Internal scaffold-disjoint metrics with DeLong and paired-bootstrap tests, then the three external arms - ZINC decoy enrichment, FDA control recovery, marine phenotypic transfer. The pre-declared selection criterion is **external marine ROC-AUC, not internal test performance**, which is what selects the random forest over every graph network.

In [ ]:
rs(6, "--check")

In [ ]:
rs(6)

### Stage 07 - consensus feature attribution

~4 h, CPU. **No GPU, and there is no GPU version of this.** Exact path-dependent TreeSHAP on a scikit-learn forest has no CUDA implementation here; the four hours are irreducible on this code path.

TreeSHAP, permutation importance and mean decrease in impurity computed independently and combined into a consensus rank, the top ten features annotated structurally, and the comparison against the graph networks' integrated-gradients attribution. Unlocks Figs. 5, 7 and 8.

In [ ]:
rs(7, "--check")

In [ ]:
rs(7)

### Stage 08 - the marine screening cascade

~25 min, CPU. **No GPU.**

All 26,754 curated marine compounds scored on the six targets by the selected random forest, then filtered: active on >= 2 targets, inside the 0.30 ECFP4 applicability domain, blood-brain barrier not excluded, PAINS-free. 26,754 -> 7 survivors, six of them staurosporine-class indolocarbazoles.

In [ ]:
rs(8, "--check")

In [ ]:
rs(8)

### Stage 09 - the convergence control  [GPU]

**~4 h on one T4 or better; ~40 h on CPU.** This is the stage a GPU is for.

Fifteen runs - five architectures x three seeds - at reference capacity, with the epoch **ceiling** at 1,000 and early-stopping patience at 25. Re-read section 1 if the word "ceiling" is doing unfamiliar work there. The published runs all stopped between epochs 34 and 86 and their marine ROC-AUC stayed in 0.394-0.538.

Resumable: a run that has written `metrics/<name>_run.json` is skipped, so a recycled runtime costs you at most the run that was in flight. The per-epoch trajectories under `metrics/` are the only thing Fig. S4 can be drawn from - no summary table substitutes for them, and the original run lost them to a scratch sweep, which is why the shipped package cannot regenerate that one figure.

In [ ]:
rs(9, "--check")

The literal command is `./run_stage.sh 09`; `EPOCHS` and `PATIENCE` from section 3c reach the trainer through the stage's environment. Expect to see `ceiling 1000 epochs, early-stopping patience 25` echoed per run, and each run to stop somewhere in the tens of epochs.

In [ ]:
if not HAS_GPU:
    print("No GPU. This is ~40 h on CPU and will not finish in a free session.")
    print("Attach a GPU, or skip stage 09 - it is an independent control, and")
    print("skipping it costs Fig. S4 and Table 8 and no main-text result.")
else:
    rs(9)

### Stage 10 - the hyperparameter-budget control  [GPU, partly]

**~5 h with the GPU arm; ~30 h on CPU.**

An exhaustive grid replacing the published 6-point random search: 81 random-forest and 96 XGBoost configurations under StratifiedKFold(5), tuned independently for all three seeds - 15,930 fits. Only the XGBoost arm uses the GPU; the random forest is scikit-learn on CPU by design.

Checkpointed per configuration under `GLIOMA_CKPT`, which section 3c put on persistent storage. **Do not delete that directory between restarts** - it is the entire resume mechanism. `ONLY_TASK=CDK6` restricts every fit to one target, which is the ten-minute rehearsal for this stage specifically.

In [ ]:
rs(10, "--check")

Rehearse first: `ONLY_TASK=CDK6` restricts every fit to a single target and turns the stage into a ten-minute end-to-end test on this platform. Then clear it and run the real grid.

In [ ]:
# ~10 min rehearsal on one target
os.environ["ONLY_TASK"] = "CDK6"
rs(10, force=True, mark=False)
os.environ.pop("ONLY_TASK", None)

In [ ]:
# the real thing: 15,930 fits, ~5 h GPU / ~30 h CPU, checkpointed per configuration
os.environ["DEVICE"] = "cuda" if HAS_GPU else "cpu"
rs(10, force=True)

### Stage 11 - regenerate every figure

~8 min, CPU. **No GPU.**

Eight main figures and four supplementary, each a vector PDF beside a 300 dpi PNG. A figure whose inputs are absent is reported `SKIPPED` with the reason rather than failing the stage. Fig. S4 needs stage 09; Figs. 5 and 8 need stage 07; Figs. 2, 3, 4, 6, S2 and S3 need the study data products or stages 02-08.

In [ ]:
rs(11, "--check")

In [ ]:
rs(11)

### Stage 12 - verify the published numbers

~2 s, CPU. **No GPU.**

Re-derives 44 published values from whatever result tables are now resolvable and asserts each against `config/published_values.json`, printing PASS / FAIL / SKIP per item. A check whose input table is absent is `SKIP`, never `PASS`. Section 10 says what to do with a FAIL.

In [ ]:
rs(12, "--check")

In [ ]:
rs(12)

---
## 7. Collect what to bring home

Everything already lives under the persistent root, because section 3c pointed
`MARINE_REPRO_OUTPUT` there. This cell just inventories it and, on Kaggle, reminds you
that nothing is kept until you **Save Version**.

In [ ]:
OUT = pathlib.Path(os.environ["MARINE_REPRO_OUTPUT"])
total = 0
for d in sorted(p for p in OUT.iterdir() if p.is_dir()):
    files = [f for f in d.rglob("*") if f.is_file()]
    size = sum(f.stat().st_size for f in files)
    total += size
    print(f"  {d.name:36s} {len(files):5d} files  {size/1e6:9.1f} MB")
print(f"  {'TOTAL':36s} {'':5s}        {total/1e6:9.1f} MB  at {OUT}")

figs = sorted((OUT / "stage11_figures").glob("*.png")) if (OUT / "stage11_figures").exists() else []
print(f"\nfigures rendered: {len(figs)} of 12 -> {[f.stem for f in figs]}")
if PLATFORM == "kaggle":
    print("\nKAGGLE: use 'Save Version' now, or /kaggle/working is discarded with the session.")

In [ ]:
ledger()

---
## 8. A multi-session running order

Kaggle stops a session at **12 hours**; Colab has no fixed cap but recycles runtimes
freely and more aggressively on free tiers. The order below assumes 12-hour sessions and
the ledger in section 3f to keep track. Every entry is resumable: re-running section 3
then the stage's cell continues rather than restarts.

| session | accelerator | stages | ~wall clock |
|---|---|---|---|
| 1 | CPU | section 3 setup, section 4 rehearsal, stages 01, 02, 04 | ~2.5 h |
| 2 | CPU | stage 03 | ~3 h |
| 3 | CPU | stage 05 group A (cache, flat arrays, fingerprints) | ~4.5 h |
| 4 | CPU | stage 05 group B, part 1 (pretraining; stops when the session does) | ~11 h |
| 5 | CPU | stage 05 group B, part 2 (resumes; skips completed runs) | ~9 h |
| 6 | CPU | stage 05 group C (baselines, ablation, final forest) | ~5.5 h |
| 7 | CPU | stage 06 | ~1.5 h |
| 8 | CPU | stage 07 | ~4 h |
| 9 | CPU | stage 08, stage 11, stage 12 | ~45 min |
| 10 | **GPU** | stage 09 | ~4 h |
| 11 | **GPU** | stage 10 | ~5 h |
| 12 | CPU | stage 11 again (picks up Figs. S4, 5, 8), stage 12 | ~10 min |

**Shortcuts that are legitimate.** Downloading the 21 study data products from the deposit
skips sessions 1-6 entirely and lets you start at stage 06. Skipping stages 09 and 10
removes sessions 10 and 11 and costs Fig. S4 and Tables 8 and 9 - no main-text result
depends on either control. Running only sessions 1, 9 and 12 reproduces the target panel,
the figures and the verification without retraining anything.

**Attach the GPU only for sessions 10 and 11.** Sessions 1-9 and 12 are CPU or
network bound; burning a GPU allocation on them buys nothing (section 2).

---
## 9. Verification, and what a FAIL means

`./run_stage.sh 12` is the one-command answer to "did this reproduce?". It re-derives 44
published values from the result tables now on disk and prints PASS, FAIL or SKIP per
item, exiting non-zero on any failure. **The reference run of the package is 44 PASS, 0
FAIL, 0 SKIP.** The science is locked: a FAIL means the regenerated value disagrees with
the manuscript, and the published value is never adjusted to make it agree.

In [ ]:
rc = rs(12, force=True)
print("\nstage 12 exit status:", rc, "(0 = every resolvable check passed)")

import csv
rep = pathlib.Path(os.environ["MARINE_REPRO_OUTPUT"]) / "stage12_verify" / "verification_report.csv"
if rep.exists():
    rows = list(csv.DictReader(open(rep)))
    bad = [r for r in rows if r.get("status", "").upper() not in ("PASS",)]
    print(f"{len(rows)} checks; {len(rows)-len(bad)} PASS")
    for r in bad:
        print("  ", r.get("status"), "|", r.get("check") or r.get("name"))

### Which numbers are expected to be bit-stable, and which are not

**Bit-stable, no matter where you run it.** Counts and shapes: the 26,754-compound library,
the 32,403 x 6 label matrix, the 45,213 decoys, 194 marine compounds with 51 actives, the
26,754 -> 7 cascade, 81 and 96 grid sizes, 15,930 fits, ten consensus features, seven
survivors, and the 300 dpi / vector-PDF figure check. A FAIL on any of these is a real
defect in the run - start by re-running stage 12 against the shipped tables alone (unset
`MARINE_REPRO_DATA` and re-run section 3c) to separate "my pipeline moved it" from "the
package is broken".

**Stable to three decimals but not bit-identical.** The seven internal macro ROC-AUCs, the
marine ROC-AUCs and Spearman coefficients, the significance tests. Stage 12 compares with
`abs_tolerance 0.0005` and `p_value_rel_tolerance 0.05` from `config/config.yaml`. BLAS
differences between platforms, thread counts and scikit-learn build variation can move the
fourth decimal; they should not move the third.

**Legitimately movable if you re-ran retrieval.** This is the important case and it is not
a failure.

* **Stage 01 queries live databases.** Open Targets re-scores associations at every
  release and CIViC gains curated evidence continuously. Section 6a of this notebook was
  run against the live APIs while it was being written, and the result is the calibration
  for what "moved" looks like:

  | | rebuilt live | published |
  |---|---|---|
  | seed genes | 186 | 186 |
  | STRING nodes | 186 | 186 |
  | STRING edges | **1,564** | **1,421** |
  | isolated seeds | 10 | 10 |
  | modules | **5** (75/66/18/8/7, 12 unassigned) | **4** (64/56/43/8, 15 unassigned) |
  | seed genes mapped to a ChEMBL target | 135 of 186 | 135 of 186 |
  | **target panel** | **EGFR, PDGFRA, PIK3CA, KDR, CDK6, IDH1** | **identical** |

  STRING has accumulated interaction evidence since the study ran; the denser graph
  splits one module. Neither moves the panel, because selection depends on centrality and
  per-hub ChEMBL compound counts rather than on the module partition. A rebuilt edge
  count above 1,421 and a fifth module are therefore expected, not a failure.
* **Stage 03 re-fetches ChEMBL**, which grows. A refreshed label matrix can legitimately
  contain more compounds than 32,403, and every downstream number then shifts slightly.
* **Stage 04 samples ZINC20**, so the decoy set is not reproducible compound-for-compound.

If a FAIL appears after a live re-fetch, the question to ask is not "is the paper wrong"
but "did the database move". Compare the regenerated intermediate against the shipped one
before concluding anything: the cached responses from the original run are in the data
deposit, and restoring them makes the pipeline deterministic again.

**`SKIP` is not `PASS`.** A check whose input table is absent is reported `SKIP`, and the
run can still be useful - but it has not verified that item. Running this notebook with
only the shipped tables gives 44 PASS because all 44 inputs ship; running it against a
fresh output directory before stage 11 gives 43 PASS and 1 SKIP, the skip being the
figure-resolution check.

---
## 10. What this notebook does not and cannot do

* **It cannot download CMNPD for you.** No public API exists and the licence forbids
  redistribution. Section 5 says where to register. Stages 02 and 03's final step are the
  only things affected.
* **It cannot make stage 05 use a GPU.** The main-benchmark trainer is CPU-only as
  shipped (section 2). Making it CUDA-capable would mean changing the code that produced
  the published numbers, which is out of scope for a reproduction.
* **It cannot make stage 07 faster.** Exact TreeSHAP on a scikit-learn forest; four hours
  is the cost.
* **It cannot guarantee a live re-fetch reproduces the paper.** It can tell you precisely
  what moved and by how much, which is section 9's job.
* **Reactome refuses some automated clients** with a Cloudflare error 1010, from some
  networks and not others. The pathway-enrichment table is the only casualty and the
  target panel does not depend on it. Do not disguise the client to get around it.